# YOLO26 Pose person counting

Simplified notebook for `input/20260329_34.mp4`. All paths, environment values, and inference hyperparameters are centralized in `data.yaml`.

The notebook counts people per processed frame using a YOLO26 Pose model, CUDA, full-frame inference (`vid_stride: 1`), and batch processing sized from the central config.


In [1]:
from __future__ import annotations

import csv
import json
import os
import random
import statistics
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import yaml


def find_notebook_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "notebooks" / "test-setup-03-run",
        cwd.parent / "test-setup-03-run",
    ]
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / "notebooks" / "test-setup-03-run")
        candidates.append(parent / "test-setup-03-run")

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "data.yaml").exists():
            return candidate

    raise FileNotFoundError("Could not find test-setup-03-run/data.yaml from the current working directory.")


NOTEBOOK_DIR = find_notebook_dir()
CONFIG_PATH = NOTEBOOK_DIR / "data.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    CONFIG: dict[str, Any] = yaml.safe_load(file) or {}


def resolve_path(value: str | Path) -> Path:
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path.resolve()
    return (NOTEBOOK_DIR / path).resolve()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def apply_environment(config: dict[str, Any]) -> None:
    path_like_keys = {"YOLO_CONFIG_DIR", "MPLCONFIGDIR", "TORCH_HOME"}
    for key, value in (config.get("environment") or {}).items():
        env_value = resolve_path(value) if key in path_like_keys else value
        if key in path_like_keys:
            ensure_dir(Path(env_value))
        os.environ[str(key)] = str(env_value)


apply_environment(CONFIG)
random.seed(int(CONFIG.get("app", {}).get("seed", 42)))

APP_CONFIG = CONFIG.get("app", {})
if APP_CONFIG.get("install_requirements", False):
    requirements_path = resolve_path(APP_CONFIG.get("requirements_file", "../../requirements.txt"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])

PATH_CONFIG = CONFIG.get("paths", {})
OUTPUT_CONFIG = CONFIG.get("output", {})

VIDEO_PATH = resolve_path(PATH_CONFIG["video"])
OUTPUT_DIR = ensure_dir(resolve_path(PATH_CONFIG.get("output_dir", "output")))
ANNOTATED_VIDEO_PATH = resolve_path(PATH_CONFIG.get("annotated_video", OUTPUT_DIR / "annotated.mp4"))
FRAME_COUNTS_CSV = resolve_path(PATH_CONFIG.get("frame_counts_csv", OUTPUT_DIR / "frame_counts.csv"))
SUMMARY_JSON = resolve_path(PATH_CONFIG.get("summary_json", OUTPUT_DIR / "summary.json"))
SNAPSHOTS_DIR = ensure_dir(resolve_path(PATH_CONFIG.get("snapshots_dir", OUTPUT_DIR / "snapshots")))

for path in [ANNOTATED_VIDEO_PATH.parent, FRAME_COUNTS_CSV.parent, SUMMARY_JSON.parent]:
    ensure_dir(path)

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Config: {CONFIG_PATH}")
print(f"Video: {VIDEO_PATH}")
print(f"Output dir: {OUTPUT_DIR}")


Notebook dir: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density
Config: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\data.yaml
Video: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\input\20260329_34.mp4
Output dir: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\output


In [2]:
import cv2
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

try:
    from ultralytics.utils import SETTINGS
    SETTINGS.update({"sync": False})
except Exception:
    pass

RUNTIME_CONFIG = CONFIG.get("runtime", {})
DEVICE = int(RUNTIME_CONFIG.get("device", 0))
REQUIRE_CUDA = bool(RUNTIME_CONFIG.get("require_cuda", True))

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required by data.yaml, but PyTorch cannot see a CUDA device.")

if torch.cuda.is_available():
    torch.cuda.set_device(DEVICE)
    allow_tf32 = bool(RUNTIME_CONFIG.get("allow_tf32", False))
    torch.backends.cuda.matmul.allow_tf32 = allow_tf32
    torch.backends.cudnn.allow_tf32 = allow_tf32

precision_mode = RUNTIME_CONFIG.get("torch_float32_matmul_precision", "highest")
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision(precision_mode)

print(f"Ultralytics: {ultralytics.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
    print(f"GPU memory free/total: {free_bytes / 1024**3:.2f} / {total_bytes / 1024**3:.2f} GiB")

try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"System RAM available/total: {ram.available / 1024**3:.2f} / {ram.total / 1024**3:.2f} GiB")
except Exception:
    print("System RAM: psutil is not available for detailed reporting.")


Ultralytics: 8.4.50
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
GPU memory free/total: 22.46 / 23.99 GiB
System RAM available/total: 91.03 / 127.84 GiB


In [3]:
def resolve_weight_reference(paths_config: dict[str, Any]) -> str:
    weight_ref = str(paths_config.get("weights", "yolo26x-pose.pt"))
    weight_path = Path(weight_ref).expanduser()

    if weight_path.is_absolute() and weight_path.exists():
        return str(weight_path.resolve())

    for search_dir in paths_config.get("weights_search_dirs", []):
        candidate = resolve_path(search_dir) / weight_ref
        if candidate.exists():
            return str(candidate.resolve())

    candidate = resolve_path(weight_ref)
    if candidate.exists():
        return str(candidate.resolve())

    # Returning the model name lets Ultralytics use its normal download/cache behavior.
    return weight_ref


if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Input video not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"OpenCV could not open video: {VIDEO_PATH}")

VIDEO_META = {
    "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "fps": float(cap.get(cv2.CAP_PROP_FPS)),
    "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
}
cap.release()

VIDEO_META["duration_sec"] = VIDEO_META["frame_count"] / VIDEO_META["fps"] if VIDEO_META["fps"] else 0.0
VIDEO_META["decoded_size_gib"] = (
    VIDEO_META["width"] * VIDEO_META["height"] * 3 * VIDEO_META["frame_count"] / 1024**3
)

WEIGHTS_REF = resolve_weight_reference(PATH_CONFIG)

print(json.dumps(VIDEO_META, indent=2))
print(f"YOLO weights reference: {WEIGHTS_REF}")
print("Decoded full-video size is shown for context; the notebook uses batched streaming to avoid exhausting RAM.")


{
  "width": 1280,
  "height": 720,
  "fps": 24.0,
  "frame_count": 43211,
  "duration_sec": 1800.4583333333333,
  "decoded_size_gib": 111.26489639282227
}
YOLO weights reference: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\yolo26x-pose.pt
Decoded full-video size is shown for context; the notebook uses batched streaming to avoid exhausting RAM.


In [4]:
INFERENCE_CONFIG = CONFIG.get("inference", {})

model = YOLO(WEIGHTS_REF, task=INFERENCE_CONFIG.get("task", "pose"))

if getattr(model, "task", None) != "pose":
    raise RuntimeError(
        f"Loaded model task is {getattr(model, 'task', None)!r}. Configure a YOLO26 Pose weight, for example yolo26x-pose.pt."
    )

if torch.cuda.is_available():
    model.to(f"cuda:{DEVICE}")

classes = INFERENCE_CONFIG.get("classes", None)
if classes == []:
    classes = None

PREDICT_ARGS = {
    "device": DEVICE,
    "imgsz": int(INFERENCE_CONFIG.get("imgsz", 1280)),
    "conf": float(INFERENCE_CONFIG.get("conf", 0.20)),
    "iou": float(INFERENCE_CONFIG.get("iou", 0.70)),
    "max_det": int(INFERENCE_CONFIG.get("max_det", 1000)),
    "classes": classes,
    "augment": bool(INFERENCE_CONFIG.get("augment", True)),
    "half": bool(INFERENCE_CONFIG.get("half", False)),
    "verbose": bool(INFERENCE_CONFIG.get("verbose", False)),
    "stream": False,
}

print("Model loaded.")
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in PREDICT_ARGS.items()}, indent=2))


Model loaded.
{
  "device": 0,
  "imgsz": 1280,
  "conf": 0.2,
  "iou": 0.7,
  "max_det": 1000,
  "classes": [
    0
  ],
  "augment": true,
  "half": false,
  "verbose": false,
  "stream": false
}


In [5]:
COUNTING_CONFIG = CONFIG.get("counting", {})
OVERLAY_CONFIG = OUTPUT_CONFIG.get("overlay", {})


def result_person_count(result) -> int:
    boxes = getattr(result, "boxes", None)
    if boxes is None:
        return 0

    count = len(boxes)
    if not COUNTING_CONFIG.get("require_keypoints", False):
        return int(count)

    keypoints = getattr(result, "keypoints", None)
    keypoint_conf = getattr(keypoints, "conf", None)
    if keypoint_conf is None:
        return int(count)

    conf_array = keypoint_conf.detach().cpu().numpy() if hasattr(keypoint_conf, "detach") else np.asarray(keypoint_conf)
    min_conf = float(COUNTING_CONFIG.get("min_keypoint_conf", 0.25))
    min_visible = int(COUNTING_CONFIG.get("min_visible_keypoints", 3))
    visible_counts = (conf_array >= min_conf).sum(axis=1)
    return int((visible_counts >= min_visible).sum())


def annotate_result(result, count: int, frame_index: int, timestamp_sec: float) -> np.ndarray:
    line_width = int(OVERLAY_CONFIG.get("thickness", 2))
    annotated = result.plot(conf=True, labels=True, boxes=True, kpt_line=True, line_width=line_width)

    if OVERLAY_CONFIG.get("enabled", True):
        label = f"People: {count} | Frame: {frame_index} | Time: {timestamp_sec:0.1f}s"
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = float(OVERLAY_CONFIG.get("font_scale", 0.8))
        thickness = int(OVERLAY_CONFIG.get("thickness", 2))
        (text_w, text_h), baseline = cv2.getTextSize(label, font, font_scale, thickness)
        x, y = 16, 32
        cv2.rectangle(annotated, (x - 8, y - text_h - 10), (x + text_w + 8, y + baseline + 8), (0, 0, 0), -1)
        cv2.putText(annotated, label, (x, y), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

    return annotated


def save_snapshot(frame: np.ndarray, frame_index: int) -> None:
    snapshot_path = SNAPSHOTS_DIR / f"frame_{frame_index:06d}.jpg"
    cv2.imwrite(str(snapshot_path), frame)


def format_seconds(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


In [6]:
def predict_batch(frames: list[np.ndarray]) -> list[Any]:
    if not frames:
        return []

    args = dict(PREDICT_ARGS)
    args["batch"] = len(frames)

    try:
        with torch.inference_mode():
            return list(model.predict(source=frames, **args))
    except torch.cuda.OutOfMemoryError:
        is_oom = True
    except RuntimeError as error:
        is_oom = "out of memory" in str(error).lower()
        if not is_oom:
            raise

    if not RUNTIME_CONFIG.get("auto_reduce_batch_on_oom", True) or len(frames) == 1:
        raise RuntimeError("CUDA out of memory while processing a single frame or auto reduction is disabled.")

    torch.cuda.empty_cache()
    midpoint = max(1, len(frames) // 2)
    return predict_batch(frames[:midpoint]) + predict_batch(frames[midpoint:])


def build_summary(counts: list[int], elapsed_sec: float, processed_frames: int) -> dict[str, Any]:
    if counts:
        p95 = float(np.percentile(np.asarray(counts), 95))
        summary_counts = {
            "min_people_in_frame": int(min(counts)),
            "max_people_in_frame": int(max(counts)),
            "mean_people_per_frame": round(float(statistics.fmean(counts)), 3),
            "median_people_per_frame": round(float(statistics.median(counts)), 3),
            "p95_people_per_frame": round(p95, 3),
        }
    else:
        summary_counts = {
            "min_people_in_frame": 0,
            "max_people_in_frame": 0,
            "mean_people_per_frame": 0.0,
            "median_people_per_frame": 0.0,
            "p95_people_per_frame": 0.0,
        }

    return {
        "app": APP_CONFIG.get("name"),
        "video": str(VIDEO_PATH),
        "weights": str(WEIGHTS_REF),
        "device": DEVICE,
        "video_meta": VIDEO_META,
        "processed_frames": processed_frames,
        "elapsed_sec": round(float(elapsed_sec), 3),
        "fps_processed": round(processed_frames / elapsed_sec, 3) if elapsed_sec else 0.0,
        "inference": PREDICT_ARGS,
        "counts": summary_counts,
        "outputs": {
            "annotated_video": str(ANNOTATED_VIDEO_PATH),
            "frame_counts_csv": str(FRAME_COUNTS_CSV),
            "summary_json": str(SUMMARY_JSON),
            "snapshots_dir": str(SNAPSHOTS_DIR),
        },
    }


def run_counting() -> dict[str, Any]:
    batch_size = max(1, int(RUNTIME_CONFIG.get("batch_size", 16)))
    vid_stride = max(1, int(INFERENCE_CONFIG.get("vid_stride", 1)))
    progress_every = max(1, int(OUTPUT_CONFIG.get("progress_every_n_frames", 720)))
    snapshot_every = int(OUTPUT_CONFIG.get("save_snapshot_every_n_frames", 720))
    expected_processed = (VIDEO_META["frame_count"] + vid_stride - 1) // vid_stride

    cap = cv2.VideoCapture(str(VIDEO_PATH))
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {VIDEO_PATH}")

    video_writer = None
    if OUTPUT_CONFIG.get("save_annotated_video", True):
        codec = str(OUTPUT_CONFIG.get("video_codec", "mp4v"))
        fourcc = cv2.VideoWriter_fourcc(*codec)
        output_fps = VIDEO_META["fps"] / vid_stride if VIDEO_META["fps"] else 24.0
        video_writer = cv2.VideoWriter(
            str(ANNOTATED_VIDEO_PATH),
            fourcc,
            output_fps,
            (VIDEO_META["width"], VIDEO_META["height"]),
        )
        if not video_writer.isOpened():
            raise RuntimeError(f"Could not open output video writer: {ANNOTATED_VIDEO_PATH}")

    csv_file = None
    csv_writer = None
    if OUTPUT_CONFIG.get("save_frame_counts", True):
        csv_file = FRAME_COUNTS_CSV.open("w", newline="", encoding="utf-8")
        csv_writer = csv.DictWriter(csv_file, fieldnames=["frame_index", "timestamp_sec", "people_count"])
        csv_writer.writeheader()

    counts: list[int] = []
    batch_frames: list[np.ndarray] = []
    batch_indices: list[int] = []
    processed_frames = 0
    frame_index = 0
    batch_number = 0
    next_progress = progress_every
    start_time = time.time()

    def flush_batch() -> None:
        nonlocal batch_number, processed_frames, next_progress
        if not batch_frames:
            return

        frames = list(batch_frames)
        indices = list(batch_indices)
        batch_frames.clear()
        batch_indices.clear()

        results = predict_batch(frames)
        if len(results) != len(indices):
            raise RuntimeError(f"Expected {len(indices)} predictions, received {len(results)}.")

        for result, source_frame, source_index in zip(results, frames, indices):
            timestamp_sec = source_index / VIDEO_META["fps"] if VIDEO_META["fps"] else 0.0
            count = result_person_count(result)
            counts.append(count)

            if csv_writer is not None:
                csv_writer.writerow({
                    "frame_index": source_index,
                    "timestamp_sec": round(timestamp_sec, 3),
                    "people_count": count,
                })

            if video_writer is not None or (snapshot_every > 0 and source_index % snapshot_every == 0):
                annotated = annotate_result(result, count, source_index, timestamp_sec)
                if annotated.shape[1] != VIDEO_META["width"] or annotated.shape[0] != VIDEO_META["height"]:
                    annotated = cv2.resize(annotated, (VIDEO_META["width"], VIDEO_META["height"]))

                if video_writer is not None:
                    video_writer.write(annotated)
                if snapshot_every > 0 and source_index % snapshot_every == 0:
                    save_snapshot(annotated, source_index)

            processed_frames += 1

        batch_number += 1
        clear_every = int(RUNTIME_CONFIG.get("empty_cuda_cache_every_batches", 0))
        if clear_every > 0 and batch_number % clear_every == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

        while processed_frames >= next_progress:
            elapsed = time.time() - start_time
            rate = processed_frames / elapsed if elapsed else 0.0
            remaining = (expected_processed - processed_frames) / rate if rate else 0.0
            print(
                f"Processed {processed_frames}/{expected_processed} frames "
                f"({rate:0.2f} FPS), ETA {format_seconds(remaining)}"
            )
            next_progress += progress_every

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_index % vid_stride == 0:
                batch_frames.append(frame)
                batch_indices.append(frame_index)
                if len(batch_frames) >= batch_size:
                    flush_batch()

            frame_index += 1

        flush_batch()
    finally:
        cap.release()
        if video_writer is not None:
            video_writer.release()
        if csv_file is not None:
            csv_file.close()

    elapsed_sec = time.time() - start_time
    summary = build_summary(counts, elapsed_sec, processed_frames)

    if OUTPUT_CONFIG.get("save_summary", True):
        with SUMMARY_JSON.open("w", encoding="utf-8") as file:
            json.dump(summary, file, indent=2)

    print("Done.")
    print(json.dumps(summary["counts"], indent=2))
    return summary


SUMMARY = run_counting()
SUMMARY


WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to singl

{'app': 'test-setup-03-run-yolo26-pose-count',
 'video': 'C:\\Users\\patrickcruz\\Documents\\Professional\\Github\\contagem-de-pessoas\\count-github-yolo-01\\notebooks\\testing\\test-setup-04-run-density\\input\\20260329_34.mp4',
 'weights': 'C:\\Users\\patrickcruz\\Documents\\Professional\\Github\\contagem-de-pessoas\\count-github-yolo-01\\notebooks\\testing\\test-setup-04-run-density\\yolo26x-pose.pt',
 'device': 0,
 'video_meta': {'width': 1280,
  'height': 720,
  'fps': 24.0,
  'frame_count': 43211,
  'duration_sec': 1800.4583333333333,
  'decoded_size_gib': 111.26489639282227},
 'processed_frames': 43211,
 'elapsed_sec': 8165.134,
 'fps_processed': 5.292,
 'inference': {'device': 0,
  'imgsz': 1280,
  'conf': 0.2,
  'iou': 0.7,
  'max_det': 1000,
  'classes': [0],
  'augment': True,
  'half': False,
  'verbose': False,
  'stream': False},
 'counts': {'min_people_in_frame': 0,
  'max_people_in_frame': 48,
  'mean_people_per_frame': 14.94,
  'median_people_per_frame': 14.0,
  'p95_p

In [7]:
from IPython.display import Video, display

print(f"Summary: {SUMMARY_JSON}")
print(f"Frame counts CSV: {FRAME_COUNTS_CSV}")
print(f"Annotated video: {ANNOTATED_VIDEO_PATH}")
print(f"Snapshots: {SNAPSHOTS_DIR}")

if ANNOTATED_VIDEO_PATH.exists():
    display(Video(str(ANNOTATED_VIDEO_PATH), embed=False, width=960))


Summary: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\output\summary.json
Frame counts CSV: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\output\frame_counts.csv
Annotated video: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\output\20260329_34_yolo26_pose_count.mp4
Snapshots: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\testing\test-setup-04-run-density\output\snapshots
